In [ ]:
import geopandas as gpd
import glob
import os
import rioxarray as rxr
import xarray as xr
from rioxarray.merge import merge_arrays

## Settings

In [ ]:
PATH = "data/GHS_LAND_*.tif"
AOI = "aoi/A00_Granice_panstwa.shp"
RASTER_CHUNKS = "auto"
WATER_PIXEL_THRESHOLD = 90
NO_DATA_VALUE = 0
OUTPUT_DIRECTORY = "output"
REPROJECT = "EPSG:2180" # None or target CRS string like "EPSG:2180"
MERGE_OUTPUT_DATA = True

## Open area of interest file

In [ ]:
gdf_aoi = gpd.read_file(AOI)
gdf_aoi

## Process raster files

In [ ]:
files = glob.glob(PATH)
files

In [ ]:
os.makedirs(OUTPUT_DIRECTORY, exist_ok=True)

In [ ]:
output_files = []

for file in files:
    print(f"Processing {file}...")
    with rxr.open_rasterio(file, cache=False, chunks=RASTER_CHUNKS) as src:
        
        print("Cropping to ROI...")
        cropped_image = src.rio.clip(
            gdf_aoi.geometry.values,
            crs=gdf_aoi.crs,
            from_disk=True
        )

        print("Reclassifying values...")
        cropped_image_reclass = xr.where(cropped_image <= WATER_PIXEL_THRESHOLD, 1, NO_DATA_VALUE)

        print("Saving metadata...")
        cropped_image_reclass.rio.write_crs(cropped_image.rio.crs, inplace=True)

        if REPROJECT:
            print("Reprojecting...")
            cropped_image_reclass = cropped_image_reclass.rio.reproject(REPROJECT)
        
        print("Exporting output file...")
        output_file = os.path.basename(file).replace(".tif", f"_reclass_{WATER_PIXEL_THRESHOLD}.tif")
        output_path = os.path.join(OUTPUT_DIRECTORY, output_file)
        output_files.append(output_path)
        
        cropped_image_reclass["band"] = cropped_image_reclass["band"].astype("uint8")
        cropped_image_reclass["x"] = cropped_image_reclass["x"].astype("float32")
        cropped_image_reclass["y"] = cropped_image_reclass["y"].astype("float32")
        cropped_image_reclass.rio.write_nodata(NO_DATA_VALUE, inplace=True)
        cropped_image_reclass.rio.to_raster(output_path, compress="lzw")
        
        print(f"Finished processing {file}\n")

In [ ]:
if MERGE_OUTPUT_DATA:
    data = []
    for file in output_files:
        data.append(rxr.open_rasterio(file, cache=False, chunks=RASTER_CHUNKS))
    
    data_merged = merge_arrays(data, nodata=NO_DATA_VALUE)
    
    output_path = os.path.join(OUTPUT_DIRECTORY, f"GHSL_LAND_E2018_GLOBE_{WATER_PIXEL_THRESHOLD}_merged.tif")
    data_merged.rio.to_raster(output_path, compress="lzw", dtype="uint8", nodata=NO_DATA_VALUE)